Nombre: Emilio Rico Hernández
Clase: 5
Challenge: Data Integration Challenge
Fecha: 2026-09-25

# Challenge 5 — Data Integration Challenge

Quiero construir una tabla analítica uniendo tres fuentes con `merge`, cuidando claves, cardinalidad y validaciones. No tengo los CSV clásicos de clientes, pedidos y productos, así que uso lo que tengo del diplomado:

- `ventas`: una fila por transacción, de `ventas_2.csv` (sin las columnas del producto).
- `productos`: catálogo que armé separando `producto_id`, `nombre_producto` y `categoria` de ese mismo archivo.
- `regiones`: el archivo `regiones.csv`.

Como aquí todas las claves tienen pareja, en el Paso 3 agrego una simulación (marcada) con un catálogo incompleto.

In [1]:
import pandas as pd

crudo = pd.read_csv("ventas_2.csv")
regiones = pd.read_csv("regiones.csv")

# ventas es la tabla de hechos; el catálogo de productos sale de las mismas columnas
ventas = crudo.drop(columns=["nombre_producto", "categoria"])
ventas.insert(0, "id_venta", range(1, len(ventas) + 1))
productos = crudo[["producto_id", "nombre_producto", "categoria"]].drop_duplicates()
print(ventas.shape, productos.shape, regiones.shape)

(6000, 8) (20, 3) (5, 3)


In [2]:
print(ventas.head(3))
print(productos.head(3))
print(regiones)

   id_venta       fecha  region_id  ... cantidad  precio_unitario     total
0         1  2023-01-01          1  ...        9          1932.73  17394.57
1         2  2023-01-01          3  ...        5          1240.04   6200.20
2         3  2023-01-01          5  ...        1          1264.57   1264.57

[3 rows x 8 columns]
   producto_id nombre_producto    categoria
0            7      Producto 7  Electrónica
1            5      Producto 5         Ropa
2            1      Producto 1  Electrónica
   region_id   region    pais
0          1    Norte  México
1          2      Sur  México
2          3   Centro  México
3          4    Bajío  México
4          5  Sureste  México


## Paso 1 — Identificar claves

In [3]:
print("id_venta único en ventas:", ventas["id_venta"].is_unique)
print("producto_id único en productos:", productos["producto_id"].is_unique)
print("region_id único en regiones:", regiones["region_id"].is_unique)
print("Nulos en las llaves foráneas de ventas:", ventas[["producto_id", "region_id"]].isna().sum().sum())

id_venta único en ventas: True
producto_id único en productos: True
region_id único en regiones: True
Nulos en las llaves foráneas de ventas: 0


- **`ventas`:** llave primaria `id_venta` (única, 6,000 filas); llaves foráneas `producto_id` y `region_id`, sin nulos.
- **`productos`:** llave primaria `producto_id`, única (20 productos).
- **`regiones`:** llave primaria `region_id`, única (5 regiones).

Ninguna llave primaria tiene duplicados, que es lo que hay que revisar antes de unir: un duplicado en el lado "uno" multiplicaría filas.

## Paso 2 — Cardinalidad

In [4]:
print("Ventas por producto (mín y máx):", ventas["producto_id"].value_counts().agg(["min", "max"]).tolist())
print("Ventas por región (mín y máx):", ventas["region_id"].value_counts().agg(["min", "max"]).tolist())
print("producto_id de ventas sin catálogo:", (~ventas["producto_id"].isin(productos["producto_id"])).sum())
print("region_id de ventas sin región:", (~ventas["region_id"].isin(regiones["region_id"])).sum())

Ventas por producto (mín y máx): [273, 327]
Ventas por región (mín y máx): [1172, 1219]
producto_id de ventas sin catálogo: 0
region_id de ventas sin región: 0


```
productos ──(1:N)── ventas ──(N:1)── regiones
PK producto_id      PK id_venta      PK region_id
                    FK producto_id
                    FK region_id
```

- **`productos` → `ventas`: uno a muchos.** Cada producto aparece en entre 273 y 327 ventas; cada venta es de un solo producto.
- **`regiones` → `ventas`: uno a muchos.** Cada región tiene entre 1,172 y 1,219 ventas.
- **`productos` y `regiones`:** solo se relacionan de forma indirecta (muchos a muchos) a través de `ventas`, que es la tabla puente.

Todos los `producto_id` y `region_id` de `ventas` existen en sus catálogos (0 huérfanos).

## Paso 3 — Joins

In [5]:
inner = ventas.merge(productos, on="producto_id", how="inner", validate="many_to_one")
inner = inner.merge(regiones, on="region_id", how="inner", validate="many_to_one")
left = ventas.merge(productos, on="producto_id", how="left", validate="many_to_one")
left = left.merge(regiones, on="region_id", how="left", validate="many_to_one")
print("Inner:", inner.shape, "| Left:", left.shape, "| ¿idénticos?", inner.equals(left))

Inner: (6000, 12) | Left: (6000, 12) | ¿idénticos? True


Con mis datos los dos joins dan lo mismo. **Simulación (solo de práctica):** quito los productos 18, 19 y 20 del catálogo para ver qué pasa cuando hay ventas sin pareja.

In [6]:
productos_inc = productos[productos["producto_id"] <= 17]
inner_x = ventas.merge(productos_inc, on="producto_id", how="inner")
left_x = ventas.merge(productos_inc, on="producto_id", how="left")
print("Filas -> ventas:", len(ventas), "| inner:", len(inner_x), "| left:", len(left_x))
print("Nulos en nombre_producto con el left join:", left_x["nombre_producto"].isna().sum())
print("Suma de total -> real:", round(ventas["total"].sum()), "| inner:", round(inner_x["total"].sum()), "| left:", round(left_x["total"].sum()))

Filas -> ventas: 6000 | inner: 5109 | left: 6000
Nulos en nombre_producto con el left join: 891
Suma de total -> real: 61984065 | inner: 56966123 | left: 61984065


Con los datos reales, inner y left dan lo mismo (6,000 filas y 12 columnas) porque todas las claves tienen pareja. Usé `validate="many_to_one"` para que pandas avise si una clave del catálogo estuviera duplicada.

En la simulación sí se nota la diferencia. El **inner** se quedó con 5,109 ventas: eliminó 891 sin ningún error, y con ellas casi $5.02 M de ingresos (8.1% del total). El **left** conservó las 6,000 ventas y la suma correcta ($61.98 M), pero dejó 891 nulos en `nombre_producto`, que muestran qué falta en el catálogo. Por eso prefiero left join.

## Paso 4 — Validación antes y después del merge

In [7]:
tablas = [("ventas (antes)", ventas), ("productos (antes)", productos), ("regiones (antes)", regiones), ("inner (después)", inner), ("left (después)", left)]
for nombre, t in tablas:
    print(f"{nombre:18s} filas: {len(t):5d} | columnas: {t.shape[1]:2d} | nulos: {t.isna().sum().sum()}")
print("id_venta únicos después del merge:", left["id_venta"].nunique())
print("Suma de total -> antes:", round(ventas["total"].sum(), 2), "| después:", round(left["total"].sum(), 2))

ventas (antes)     filas:  6000 | columnas:  8 | nulos: 0
productos (antes)  filas:    20 | columnas:  3 | nulos: 0
regiones (antes)   filas:     5 | columnas:  3 | nulos: 0
inner (después)    filas:  6000 | columnas: 12 | nulos: 0
left (después)     filas:  6000 | columnas: 12 | nulos: 0
id_venta únicos después del merge: 6000
Suma de total -> antes: 61984064.58 | después: 61984064.58


Después de las dos uniones `ventas` sigue con 6,000 filas: no se perdió ni se duplicó ninguna. Las columnas pasaron de 8 a 12 (+`nombre_producto`, `categoria`, `region` y `pais`), `id_venta` sigue siendo única (6,000 valores) y no se generaron valores faltantes (0 nulos). La suma de `total` es la misma antes y después ($61,984,064.58).

## Paso 5 — Tabla analítica

In [8]:
tabla = left
tabla.head()

,id_venta,fecha,region_id,vendedor,producto_id,cantidad,precio_unitario,total,nombre_producto,categoria,region,pais
0,1,2023-01-01,1,Vendedor_4,7,9,1932.73,17394.57,Producto 7,Electrónica,Norte,México
1,2,2023-01-01,3,Vendedor_10,5,5,1240.04,6200.20,Producto 5,Ropa,Centro,México
2,3,2023-01-01,5,Vendedor_6,1,1,1264.57,1264.57,Producto 1,Electrónica,Sureste,México
3,4,2023-01-01,4,Vendedor_9,14,3,2949.45,8848.35,Producto 14,Deportes,Bajío,México
4,5,2023-01-01,2,Vendedor_6,17,4,1593.14,6372.56,Producto 17,Ropa,Sur,México


**¿Qué producto genera mayor ingreso?**

In [9]:
tabla.groupby(["nombre_producto", "categoria"])["total"].sum().sort_values(ascending=False).head(3).round(2)

nombre_producto  categoria
Producto 13      Deportes     5657119.42
Producto 14      Deportes     5234997.57
Producto 4       Ropa         5115906.98
Name: total, dtype: float64

**¿Qué categoría tiene más transacciones?**

In [10]:
tabla.groupby("categoria")["id_venta"].count().sort_values(ascending=False)

categoria
Deportes       2118
Ropa           1555
Electrónica    1160
Alimentos       850
Hogar           317
Name: id_venta, dtype: int64

**¿Qué región genera más ingreso?**

In [11]:
tabla.groupby("region")["total"].sum().sort_values(ascending=False).round(2)

region
Sur        12850555.26
Centro     12737707.39
Norte      12272701.72
Sureste    12096084.01
Bajío      12027016.20
Name: total, dtype: float64

Con la tabla final (una fila por venta, ya con producto, categoría y región) respondí tres preguntas:

1. **Producto con mayor ingreso:** el Producto 13 (Deportes), con $5.66 M, seguido del Producto 14 ($5.23 M) y el Producto 4 ($5.12 M).
2. **Categoría con más transacciones:** Deportes, con 2,118 (35% de las 6,000), y luego Ropa con 1,555. Hogar es la que menos tiene (317).
3. **Región con más ingreso:** el Sur ($12.85 M), aunque por poco: las cinco regiones venden entre $12.0 M y $12.9 M.

## Preguntas de reflexión

1. **¿Qué diferencias observaste entre inner y left join?** Con los datos reales, ninguna: los dos dan 6,000 filas y 12 columnas. Con el catálogo incompleto de la simulación, el inner eliminó 891 ventas y el left las conservó con nulos en las columnas del catálogo.

2. **¿Aparecieron registros sin correspondencia?** En los datos reales no (0 huérfanos). Solo en la simulación, con los productos 18, 19 y 20.

3. **¿El número de filas aumentó inesperadamente?** No, siguió en 6,000. Habría aumentado con una clave duplicada en `productos` o `regiones`, y `validate="many_to_one"` lo habría detenido con un error.

4. **¿Existía alguna relación muchos a muchos?** No directamente: las dos relaciones que uní son de uno a muchos. Entre productos y regiones hay una indirecta, resuelta por la tabla `ventas`.

## Conclusiones

1. Las tres tablas se integran sin pérdidas ni duplicados: 6,000 ventas antes y después, `id_venta` única, 0 nulos generados y la misma suma de ventas ($61.98 M).
2. La tabla final responde lo que ninguna tabla sola podía: el Producto 13 es el de mayor ingreso, Deportes tiene más transacciones y el Sur vende más, aunque por poco.
3. Hay que revisar claves y cardinalidad antes de unir, porque los errores no avisan: en la simulación, el inner perdió 891 ventas sin mensaje. Prefiero left join con `validate` y comparar filas y sumas antes y después.